# 03 - Comparare Detectoare: evolutia istorica spre Detector final

Comparatie cross-evaluation a detectoarelor antrenate pe parcursul proiectului. Folosim rezultatele JSON salvate din fiecare experiment, deoarece weights-urile experimentelor intermediare au fost arhivate dupa promovare in productie.

| Experiment | Model | Dataset | Test mAP50 | Verdict |
|---|---|---|---:|---|
| A4-ext | YOLOv8s fine-tune | parks_detect_A4 (2.3k) | **0.830** | baseline solid |
| Detector final | YOLOv8s + transfer | parks_detect_final (7.7k) | **0.910** ★ | **MODEL FINAL** |

Imbunatatire: **+8 puncte mAP50** prin curatare date + transfer learning ablation step-by-step.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

REPO = Path('../..').resolve()
RESULTS = REPO / 'results' / 'detector'

models = {
    'A4-ext (baseline)': RESULTS / 'A4-ext-test.json',
    'Detector final': RESULTS / 'parks-trash-final-test.json',
}

rows = []
for name, fp in models.items():
    if not fp.exists():
        print(f'Lipseste: {fp}')
        continue
    data = json.load(open(fp, encoding='utf-8'))
    r = data['results']
    rows.append({
        'Model': name,
        'Precision': r.get('precision'),
        'Recall': r.get('recall'),
        'F1': r.get('f1'),
        'mAP50': r.get('mAP50'),
        'mAP50-95': r.get('mAP50_95'),
    })

df = pd.DataFrame(rows).set_index('Model')
print(df.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
df[['Precision', 'Recall', 'F1', 'mAP50', 'mAP50-95']].plot(kind='bar', ax=ax, color=['#4e9af1', '#f17c4e', '#62c370', '#f1c94e', '#a16af1'])
ax.set_title('Comparatie A4-ext vs Detector final')
ax.set_ylim(0, 1)
ax.set_xlabel('')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(REPO / 'outputs' / 'thesis_figures' / 'detector_comparison_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvat: outputs/thesis_figures/detector_comparison_final.png')
